In [ ]:
# 1. InternVL3-8B Non-Quantized Setup and Imports
# This notebook tests non-quantized InternVL3-8B performance using specialized memory optimization
# to handle the large vision encoder without OOM errors.

# Enable autoreload for module changes
%load_ext autoreload
%autoreload 2

# Standard library imports
import sys
import warnings
from datetime import datetime
from pathlib import Path

# Add current directory to path to ensure proper module resolution
notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

# Third-party imports
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rich import print as rprint
from rich.console import Console

# Project-specific imports
from common.batch_analytics import BatchAnalytics
from common.batch_processor import BatchDocumentProcessor
from common.batch_reporting import BatchReporter
from common.batch_visualizations import BatchVisualizer
from common.evaluation_metrics import load_ground_truth
from common.extraction_parser import discover_images
from common.internvl3_8b_memory_optimizer import load_internvl3_8b_optimized
from models.document_aware_internvl3_processor import (
    DocumentAwareInternVL3HybridProcessor,
)

print("✅ All imports loaded successfully")
print("✅ InternVL3-8B Memory Optimizer imported successfully") 
print("✅ Specialized memory management enabled for large vision encoder")
print(f"📂 Working directory: {notebook_dir}")
print("🔬 NON-QUANTIZED InternVL3-8B: Testing with memory optimization")
warnings.filterwarnings('ignore')

In [ ]:
# 2. Aggressive Memory Cleanup for InternVL3-8B
# CRITICAL: Specialized cleanup before loading the large vision encoder

from common.internvl3_8b_memory_optimizer import InternVL3_8B_MemoryManager

rprint("[bold red]🧹 AGGRESSIVE MEMORY CLEANUP FOR INTERNVL3-8B[/bold red]")
rprint("[yellow]Performing specialized cleanup for large vision encoder...[/yellow]")
rprint("[cyan]💡 This addresses the disproportionately large memory requirements of InternVL3-8B[/cyan]")

# Initialize memory manager and perform cleanup
memory_manager = InternVL3_8B_MemoryManager(verbose=True)
memory_manager.aggressive_cleanup_for_8b()

# Create initial memory checkpoint
initial_checkpoint = memory_manager.create_memory_checkpoint("notebook_start")

rprint("[green]✅ Aggressive cleanup complete - ready for InternVL3-8B loading[/green]")
rprint("[dim]📋 Next: Configure settings and load model with memory monitoring[/dim]")

In [ ]:
# 3. Configuration - InternVL3-8B Non-Quantized Settings
# Initialize console and environment configuration

console = Console()

# Environment-specific base paths
ENVIRONMENT_BASES = {
    'sandbox': '/home/jovyan/nfs_share/tod',
    'efs': '/efs/shared/PoC_data'
}
base_data_path = ENVIRONMENT_BASES['sandbox']

CONFIG = {
    # Model settings - CRITICAL: InternVL3-8B path
    'MODEL_PATH': '/home/jovyan/nfs_share/models/InternVL3-8B',
    # 'MODEL_PATH': '/efs/shared/PTM/InternVL3-8B',
    
    # Batch settings
    'DATA_DIR': f'{base_data_path}/evaluation_data',
    'GROUND_TRUTH': f'{base_data_path}/evaluation_data/ground_truth.csv',
    'OUTPUT_BASE': f'{base_data_path}/output',
    'MAX_IMAGES': None,  # None for all, or set limit for testing
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Inference and evaluation mode
    'INFERENCE_ONLY': False,  # Set to True for inference-only mode
    
    # Verbosity control
    'VERBOSE': True,
    'SHOW_PROMPTS': True,
    
    # InternVL3-8B optimization settings - NON-QUANTIZED with MEMORY OPTIMIZATION
    'USE_QUANTIZATION': False,  # TESTING: Non-quantized with memory optimization
    'DEVICE_MAP': 'auto',
    'MAX_NEW_TOKENS': 600,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True,
    # Flash Attention: V100 compatible setting
    'USE_FLASH_ATTN': False  # V100 compatible default
}

# Make GROUND_TRUTH conditional based on INFERENCE_ONLY mode
if CONFIG['INFERENCE_ONLY']:
    CONFIG['GROUND_TRUTH'] = None

# InternVL3 prompt configuration
PROMPT_CONFIG = {
    'detection_file': 'prompts/document_type_detection.yaml',
    'detection_key': 'detection',
    'extraction_files': {
        'INVOICE': 'prompts/internvl3_prompts.yaml',
        'RECEIPT': 'prompts/internvl3_prompts.yaml', 
        'BANK_STATEMENT': 'prompts/internvl3_prompts.yaml'
    },
    'extraction_keys': {
        'INVOICE': 'invoice',
        'RECEIPT': 'receipt',
        'BANK_STATEMENT': 'bank_statement'
    }
}

# Field list required for DocumentAwareInternVL3HybridProcessor
UNIVERSAL_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "STATEMENT_DATE_RANGE",
    "LINE_ITEM_DESCRIPTIONS", "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES",
    "LINE_ITEM_TOTAL_PRICES", "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT",
    "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID", "TRANSACTION_AMOUNTS_RECEIVED",
    "ACCOUNT_BALANCE"
]

print("✅ Configuration set up successfully")
print(f"📂 Evaluation data: {CONFIG['DATA_DIR']}")
print(f"🤖 Model path: {CONFIG['MODEL_PATH']}")
print(f"⚙️  Quantization: {'ENABLED (8-bit)' if CONFIG['USE_QUANTIZATION'] else 'DISABLED (full precision)'}")
print("🔬 TESTING: InternVL3-8B non-quantized with specialized memory optimization")

In [ ]:
# 4. Load InternVL3-8B with Specialized Memory Optimization
# Uses sequential component loading and memory monitoring to handle the large vision encoder without OOM errors

rprint("[bold green]🚀 Loading InternVL3-8B with Memory Optimization...[/bold green]")
rprint("[cyan]🔬 Testing: Non-quantized 8B model with specialized memory management[/cyan]")
rprint("[cyan]📖 Bypassing quantization force-override from internvl3_model_loader.py[/cyan]")

try:
    # Convert torch_dtype string to torch dtype
    dtype_map = {
        "bfloat16": torch.bfloat16,
        "float16": torch.float16,
        "float32": torch.float32
    }
    torch_dtype_obj = dtype_map.get(CONFIG['TORCH_DTYPE'], torch.bfloat16)
    
    # Use specialized InternVL3-8B loading with memory optimization
    model, tokenizer = load_internvl3_8b_optimized(
        model_path=CONFIG['MODEL_PATH'],
        torch_dtype=torch_dtype_obj,
        low_cpu_mem_usage=CONFIG['LOW_CPU_MEM_USAGE'],
        use_flash_attn=CONFIG['USE_FLASH_ATTN'],
        verbose=CONFIG['VERBOSE']
    )
    
    # Set generation parameters
    model.config.max_new_tokens = CONFIG['MAX_NEW_TOKENS']
    
    # Display model information
    rprint("[green]✅ InternVL3-8B model and tokenizer loaded successfully![/green]")
    
    # GPU memory status
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        rprint(f"[blue]📊 GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved, {total:.0f}GB total[/blue]")
        rprint(f"[blue]🔍 Memory usage: {(allocated/total*100):.1f}%[/blue]")
    
    # Model parameters
    param_count = sum(p.numel() for p in model.parameters())
    rprint(f"[blue]🔢 Model parameters: {param_count:,}[/blue]")
    rprint(f"[blue]🎯 Data type: {model.dtype}[/blue]")
    rprint(f"[blue]🖥️  Device: {next(model.parameters()).device}[/blue]")
    
    # Initialize the hybrid processor with loaded model components
    rprint("[cyan]🔧 Initializing document-aware processor...[/cyan]")
    hybrid_processor = DocumentAwareInternVL3HybridProcessor(
        field_list=UNIVERSAL_FIELDS,
        model_path=CONFIG['MODEL_PATH'],
        debug=CONFIG['VERBOSE'],
        pre_loaded_model=model,
        pre_loaded_tokenizer=tokenizer,
        prompt_config=PROMPT_CONFIG
    )
    
    rprint("[bold green]✅ InternVL3-8B NON-QUANTIZED model ready for document processing[/bold green]")
    rprint("[yellow]🔬 SUCCESS: Large vision encoder loaded without quantization![/yellow]")
    rprint("[yellow]🎉 Memory optimization successfully handled InternVL3-8B requirements[/yellow]")
    
except Exception as e:
    rprint(f"[red]❌ Error loading InternVL3-8B: {e}[/red]")
    rprint("[yellow]💡 This may indicate that additional memory optimization is needed[/yellow]")
    
    # Show memory report even on failure
    if 'memory_manager' in locals():
        memory_manager.print_memory_report()
    
    raise

In [ ]:
# 5. Testing Instructions and Next Steps
# This notebook provides the foundation for testing InternVL3-8B without quantization

print("✅ InternVL3-8B Non-Quantized Setup Complete!")
print("")
print("🎯 SUCCESS INDICATORS:")
print("  ✅ Model loads without OOM errors")
print("  ✅ Clean, coherent responses (not gibberish)")
print("  ✅ Accuracy comparable to quantized version")
print("  ✅ Memory usage stays within V100 limits")
print("")
print("📋 TO COMPLETE FULL TESTING:")
print("  1. Run cells 1-4 above to verify model loading")
print("  2. Add image discovery and batch processing cells")
print("  3. Copy cells 6-11 from ivl3_batch_non_quantized.ipynb")
print("  4. Monitor memory throughout process")
print("")
print("🔍 IF OOM STILL OCCURS:")
print("  - Check: memory_manager.print_memory_report()")
print("  - Reduce MAX_IMAGES for initial testing")
print("  - Verify no other models in memory")
print("  - Consider CPU offloading for vision components")
print("")
print("🚀 READY TO TEST NON-QUANTIZED INTERNVL3-8B!")

# Show current memory status
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"📊 Current GPU Memory: {allocated:.2f}GB / {total:.0f}GB ({allocated/total*100:.1f}%)")
    
# Print memory optimization report if available
if 'memory_manager' in locals():
    print("\n📊 Memory Optimization Report:")
    memory_manager.print_memory_report()